# Tuần 2 · So sánh hai cách khởi tạo LSTM
Phiên bản 2.2. E3 và E4 dùng cùng kiến trúc, dữ liệu, seed, thứ tự lô và ngân sách 6 epoch.
Chỉ E4 chép các vector Word2Vec vào lớp Embedding trước khi huấn luyện. Cả hai lớp Embedding đều được cập nhật.
Viết giả thuyết có ngày trước khi chạy E4. So sánh hai mô hình bằng macro-F1 và F1 từng nhãn trên validation.

Dữ liệu: **Vietnamese AI Feedback OOD 3125**, giấy phép CC BY 4.0. Dùng các tệp CSV đã chia sẵn từ gói lớp. Xem README để đặt tệp và chạy. Tuần 1–2 chỉ đọc train và validation. Nhận tệp test của lớp vào tuần 3.

In [ ]:
import hashlib
import io
import json
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

LABELS = ["negative", "neutral", "positive"]
DATA_VERSION = "ood3125-de7a66b-group42-v1"
CHECKSUMS = {'train': '17e2029dcf20ba77ef95b62b77e4a0458eabea1d36cb72e0794f790340ea1201', 'validation': 'c8148193a9d14963fbc25837f5bb6277bb907d36d1811d2cc25ea6e935847d0c', 'test': '4bee1b48b6dc277742f4d9591c602a720d4a483646e57786df40bb47f2cb7ef7'}


def text_tokens(text):
    """Tokenize word-like spans after Unicode normalization and case folding."""
    return re.findall(r"\w+", unicodedata.normalize("NFC", text).casefold())


def read_split(split):
    filename = f"ood3125_{split}.csv"
    path = Path("data") / filename
    if not path.is_file():
        path = Path(filename)
    if not path.is_file():
        raise FileNotFoundError(f"Đặt {filename} trong data/ hoặc thư mục làm việc.")
    content = path.read_bytes()
    if hashlib.sha256(content).hexdigest() != CHECKSUMS[split]:
        raise ValueError(f"{filename} không đúng bản chia của lớp DCT1243.")
    frame = pd.read_csv(io.BytesIO(content))
    frame = frame.rename(columns={"sentence": "text"})
    frame["label"] = frame["label"].map(dict(enumerate(LABELS)))
    return frame


def scores(gold, predicted):
    per_label = f1_score(gold, predicted, labels=LABELS, average=None, zero_division=0)
    return {"accuracy": accuracy_score(gold, predicted), "macro_f1": per_label.mean(),
            **{f"f1_{label}": value for label, value in zip(LABELS, per_label)}}

from gensim.models import Word2Vec

MODEL_SEED = 42
EMBED_DIM = 100


def fit_word2vec(tokenized_train):
    return Word2Vec(tokenized_train, vector_size=EMBED_DIM, window=5, min_count=2,
                    sg=1, negative=5, epochs=20, seed=MODEL_SEED, workers=1)

import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader, TensorDataset

MAX_TOKENS = 50
HIDDEN_DIM = 64
MAX_EPOCHS = 6
torch.set_num_threads(2)


def encode(texts, vocabulary):
    rows, lengths = [], []
    for text in texts:
        ids = [vocabulary.get(token, 1) for token in text_tokens(text)[:MAX_TOKENS]]
        lengths.append(max(1, len(ids)))
        rows.append(ids + [0] * (MAX_TOKENS - len(ids)))
    return torch.tensor(rows, dtype=torch.long), torch.tensor(lengths)


class SentimentLSTM(nn.Module):
    def __init__(self, vocabulary_size, initial_embeddings=None):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, EMBED_DIM, padding_idx=0)
        if initial_embeddings is not None:
            with torch.no_grad():
                self.embedding.weight.copy_(initial_embeddings)
        self.lstm = nn.LSTM(EMBED_DIM, HIDDEN_DIM, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(HIDDEN_DIM, len(LABELS))

    def forward(self, token_ids, lengths):
        embeddings = self.embedding(token_ids)  # [batch, time, embedding]
        packed = pack_padded_sequence(embeddings, lengths, batch_first=True,
                                      enforce_sorted=False)
        _, (hidden, _) = self.lstm(packed)  # [1, batch, hidden]
        return self.classifier(self.dropout(hidden[-1]))  # [batch, 3]


def predict_lstm(model, texts, vocabulary):
    model.eval()
    token_ids, lengths = encode(texts, vocabulary)
    batches = DataLoader(TensorDataset(token_ids, lengths), batch_size=128)
    with torch.no_grad():
        indices = torch.cat([model(ids, lens).argmax(1) for ids, lens in batches])
    return np.array(LABELS)[indices.numpy()]

from copy import deepcopy


def fit_lstm(train, validation, vocabulary, initial_embeddings):
    torch.manual_seed(MODEL_SEED)
    model = SentimentLSTM(len(vocabulary), initial_embeddings)
    train_ids, train_lengths = encode(train["text"], vocabulary)
    train_labels = torch.tensor(train["label"].map(LABELS.index).to_numpy())
    batches = DataLoader(TensorDataset(train_ids, train_lengths, train_labels),
                         batch_size=64, shuffle=True,
                         generator=torch.Generator().manual_seed(MODEL_SEED))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    history = []
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        total_loss = 0.0
        for ids, lengths, gold in batches:
            optimizer.zero_grad()
            logits = model(ids, lengths)
            loss = loss_fn(logits, gold)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(gold)
        predicted = predict_lstm(model, validation["text"], vocabulary)
        value = scores(validation["label"], predicted)["macro_f1"]
        history.append({"epoch": epoch, "train_loss": total_loss / len(train),
                        "macro_f1": value})
        if epoch == 1 or value > best_f1:
            best_f1, best_epoch = value, epoch
            best_state = deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_epoch

def checkpoint_digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

In [ ]:
STUDENT_ID = "3124410066"
if not STUDENT_ID.isascii() or not STUDENT_ID.isdigit() or len(STUDENT_ID) < 3:
    raise ValueError("Điền MSSV gồm ít nhất ba chữ số rồi chạy lại ô này.")
PERSONAL_SEED = int(STUDENT_ID[-3:])
print("MSSV:", STUDENT_ID, "| Seed lấy mẫu:", PERSONAL_SEED)

In [ ]:
train, validation = read_split("train"), read_split("validation")
word2vec = fit_word2vec([text_tokens(text) for text in train["text"]])
vocabulary = {"<pad>": 0, "<unk>": 1}
vocabulary.update({token: i for i, token in enumerate(word2vec.wv.index_to_key, 2)})
torch.manual_seed(MODEL_SEED)
random_embeddings = SentimentLSTM(len(vocabulary)).embedding.weight.detach().clone()
print("Vocab / embedding / hidden:", len(vocabulary), EMBED_DIM, HIDDEN_DIM)

## C3 · LSTM và giả thuyết
Đọc `forward` và `fit_lstm`. Trong báo cáo, ghi kích thước tensor ở ba vị trí
(Embedding, trạng thái ẩn cuối, đầu ra) và giải thích vai trò của độ dài thật trong xử lý padding.
Chạy E3. Ghi giả thuyết và ngày viết tại ô văn bản bên dưới trước khi chạy E4.

In [ ]:
e3, history_e3, epoch_e3 = fit_lstm(train, validation, vocabulary, None)

**Giả thuyết và ngày viết** — điền cơ chế, dự đoán đo được và điều kiện không ủng hộ dự đoán.

### Tạo ma trận khởi tạo E4
Sao chép ma trận ngẫu nhiên của E3. Trong vòng lặp, thay dòng của token `token` bằng vector Word2Vec
tương ứng. Hai dòng `<pad>` và `<unk>` dùng chung giá trị khởi tạo của E3.

**Thực hành trước khi làm mục này.** Xem [hướng dẫn Skip-gram](DCT1243_Skipgram_HuongDan.pdf) và chạy [notebook minh họa](DCT1243_Skipgram_SinhVien.ipynb) đi kèm. Nếu Colab không mở được liên kết tương đối, hãy mở hai tệp từ gói đã giải nén.

In [ ]:
w2v_embeddings = random_embeddings.clone()
for token, index in vocabulary.items():
    if token in word2vec.wv:
        w2v_embeddings[index] = torch.tensor(word2vec.wv[token])

In [ ]:
probe = word2vec.wv.index_to_key[0]
assert torch.allclose(w2v_embeddings[vocabulary[probe]],
                      torch.tensor(word2vec.wv[probe])), "Kiểm tra dòng Word2Vec."
assert torch.equal(w2v_embeddings[:2], random_embeddings[:2])
e4, history_e4, epoch_e4 = fit_lstm(train, validation, vocabulary, w2v_embeddings)
models = {"E3": e3, "E4": e4}
epochs = {"E3": epoch_e3, "E4": epoch_e4}
histories = {"E3": history_e3, "E4": history_e4}
validation_scores = pd.DataFrame.from_dict({
    name: scores(validation["label"], predict_lstm(model, validation["text"], vocabulary))
    for name, model in models.items()
}, orient="index")
validation_scores["epoch"] = pd.Series(epochs)
display(pd.concat(histories, names=["model"]).reset_index(level=1, drop=True))
display(validation_scores.round(4))

### Khóa checkpoint
Trong từng mô hình, chọn epoch có macro-F1 validation cao nhất và lấy epoch sớm hơn nếu bằng nhau.
Chọn E3 hoặc E4 theo macro-F1 validation chưa làm tròn, ưu tiên E3 nếu bằng nhau.
Ghi một nhận xét về kết quả so với giả thuyết. Tải checkpoint và tệp quyết định xuống cùng lúc.

In [ ]:
chosen = validation_scores["macro_f1"].idxmax()
checkpoint = Path(f"{STUDENT_ID}_lstm.pt")
torch.save({"vocabulary": vocabulary, "weights": models[chosen].state_dict()}, checkpoint)
decision = {"dataset": DATA_VERSION, "student_id": STUDENT_ID, "model": chosen, "epoch": epochs[chosen],
            "validation_macro_f1": float(validation_scores.loc[chosen, "macro_f1"]),
            "checkpoint_sha256": checkpoint_digest(checkpoint)}
Path(f"{STUDENT_ID}_decision.json").write_text(json.dumps(decision), encoding="utf-8")
print("QUYET_DINH", json.dumps(decision))